# Time Series

Time series methods analyze data ordered through time. In finance, prices, returns, rates, volumes, and spreads are all time series.

Abbreviations used in this notebook:

- **AR**: Autoregressive model.
- **AR(1)**: First-order autoregressive model.
- **MA**: Moving Average.
- **ACF**: Autocorrelation Function.
- **EWMA**: Exponentially Weighted Moving Average.
- **CHF**: Swiss franc, used only as an illustrative currency.

## 1. Intuition

Time series analysis asks whether today contains information about tomorrow. Returns often have weak autocorrelation, but volatility tends to cluster: calm periods and turbulent periods can persist.

## 2. Mathematics

AR(1) model:

$$
R_t = c + \phi R_{t-1} + \epsilon_t
$$

Lag-k autocorrelation:

$$
\rho_k = Corr(R_t, R_{t-k})
$$

EWMA volatility:

$$
\sigma_t^2 = \lambda \sigma_{t-1}^2 + (1 - \lambda)R_t^2
$$

Where:
- `R_t` = return at time `t`.
- `c` = constant term.
- `phi` = autoregressive coefficient.
- `epsilon_t` = residual shock at time `t`.
- `rho_k` = autocorrelation at lag `k`.
- `sigma_t^2` = variance estimate at time `t`.
- `lambda` = EWMA decay factor.


## 3. Implementation

We estimate lagged return autocorrelation and rolling volatility from synthetic asset returns.

In [ ]:
import importlib.util
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

project_root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "GUIDELINES.md").exists())
helper_path = project_root / "04_quantitative_methods" / "quant_utils.py"
spec = importlib.util.spec_from_file_location("quant_utils", helper_path)
quant_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(quant_utils)

plt.style.use("seaborn-v0_8-whitegrid")
returns = quant_utils.generate_return_sample()

asset = returns["asset"]
ts = pd.DataFrame({"return": asset})
ts["lag_1"] = ts["return"].shift(1)
ts["rolling_21d_vol"] = ts["return"].rolling(21).std() * np.sqrt(252)
ts["ewma_vol"] = ts["return"].ewm(alpha=1-0.94).std() * np.sqrt(252)

ar_model = quant_utils.ordinary_least_squares(ts.dropna()["return"].to_numpy(), ts.dropna()["lag_1"].to_numpy())
alpha, phi = ar_model["coefficients"]

pd.Series({"ar_intercept": alpha, "ar_1_phi": phi, "ar_r_squared": ar_model["r_squared"]}).to_frame("value")

In [ ]:
autocorrelations = pd.Series({f"lag_{lag}": asset.autocorr(lag=lag) for lag in range(1, 11)})
autocorrelations.to_frame("autocorrelation")

## 4. Visualization

Time series plots show regime changes, volatility clustering, and lag relationships.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 9), sharex=True)
ts["return"].plot(ax=axes[0], color="#2f6f8f")
axes[0].set_title("Daily Returns")
axes[0].yaxis.set_major_formatter(lambda x, pos: f"{x:.1%}")

ts[["rolling_21d_vol", "ewma_vol"]].plot(ax=axes[1], color=["#2f6f8f", "#9a6b2f"])
axes[1].set_title("Volatility Estimates")
axes[1].yaxis.set_major_formatter(lambda x, pos: f"{x:.0%}")

axes[2].bar(range(1, 11), autocorrelations.values, color="#2f6f8f")
axes[2].axhline(0, color="black", linewidth=1)
axes[2].set_title("Return Autocorrelation by Lag")
axes[2].set_xlabel("Lag")
plt.tight_layout(); plt.show()

## 5. Application

Time series tools are used for volatility forecasting, risk monitoring, signal testing, and checking whether strategy returns are independent through time.

In [ ]:
latest = ts.dropna().iloc[-1]
next_return_forecast = alpha + phi * latest["return"]
print(f"Latest daily return: {latest['return']:.2%}")
print(f"AR(1) next-day return forecast: {next_return_forecast:.3%}")
print(f"Latest EWMA annualized volatility: {latest['ewma_vol']:.1%}")

## 6. Reflection

- Financial returns often have low autocorrelation.
- Volatility clustering is usually stronger than return predictability.
- Rolling windows are simple but depend on window choice.
- Time series models must respect ordering; shuffling destroys information.

Questions to answer after running the notebook:

1. Is lag-1 autocorrelation meaningful here?
2. Does volatility appear clustered?
3. Why might EWMA react faster than rolling volatility?
4. What would you test before using an AR model in a strategy?